In [19]:
import pandas as pd
import tqdm
import bte

Notebook to merge UShER tree strain IDs with metadata, generate TSV files, and run MetaFitch to infer the date and country of internal nodes using Fitch's algorithm.

In [21]:
tree = bte.MATree(pb_file="/Users/reem/2026_updated_tree.pb")

Finished 'from_pb' in 219.7722 seconds


In [22]:
all_nodes = list(tree.depth_first_expansion())
nodes = []
for node in all_nodes:
    nodes.append({
        "strain": node.id})
all_nodes = pd.DataFrame(nodes)
all_nodes.head()
   

,strain
0,node_1
1,Yunnan/0306-466/2020|EPI_ISL_429239|2020-03-06
2,Japan/DP0803/2020|EPI_ISL_416630|2020-02-17
3,node_2
4,England/LEED-2A8B52/2020|EPI_ISL_539074|2020-0...


In [23]:
usher_metadata =  pd.read_csv("/Users/reem/Downloads/gisaidAndPublic.2026-01-26.gisaidNames.metadata.tsv", sep="\t")
usher_metadata.head()

/var/folders/bt/jpy5j9ms7pb6p6hhqzvpjffw0000gn/T/ipykernel_94098/4235234193.py:1: DtypeWarning: Columns (1,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  usher_metadata =  pd.read_csv("/Users/reem/Downloads/gisaidAndPublic.2026-01-26.gisaidNames.metadata.tsv", sep="\t")


,strain,genbank_accession,date,country,host,completeness,length,Nextstrain_clade,pangolin_lineage,Nextstrain_clade_usher,pango_lineage_usher
0,186951/USA/2023|PX732842.1|2023,PX732842.1,2023,USA,NaN,NaN,29649.0,23A (XBB.1.5),XBB.1.5,23A (XBB.1.5),XBB.1.5_17124
1,Guangdong/HKU-SZ-002/2020|EPI_ISL_406030|2020-...,MN938384.1,2020-01-10,China,NaN,NaN,29838.0,19B,A,19B,A
2,Shenzhen/HKU-SZ-005/2020|EPI_ISL_405839|2020-0...,MN975262.1,2020-01-11,China,NaN,NaN,29891.0,19B,A,19B,A
3,Wuhan/WHU01/2020|EPI_ISL_406716|2020-01-02,MN988668.1,2020-01-02,China,NaN,NaN,29881.0,19A,B,19A,B
4,Wuhan/WHU02/2020|EPI_ISL_406717|2020-01-02,MN988669.1,2020-01-02,China,NaN,NaN,29881.0,19A,B,19A,B


In [25]:
usher_metadata.shape

(17245272, 11)

In [45]:

all_nodes.head()

,strain
0,node_1
1,Yunnan/0306-466/2020|EPI_ISL_429239|2020-03-06
2,Japan/DP0803/2020|EPI_ISL_416630|2020-02-17
3,node_2
4,England/LEED-2A8B52/2020|EPI_ISL_539074|2020-0...


In [24]:
len(all_nodes)

21244182

In [52]:
# Merge date and country form metadata with bte_nodes on the strain col
metafitch_meta = all_nodes.merge(usher_metadata[["strain","country"]], on="strain", how="left")
metafitch_meta.head()

,strain,country
0,node_1,NaN
1,Yunnan/0306-466/2020|EPI_ISL_429239|2020-03-06,China
2,Japan/DP0803/2020|EPI_ISL_416630|2020-02-17,NaN
3,node_2,NaN
4,England/LEED-2A8B52/2020|EPI_ISL_539074|2020-0...,England


In [53]:
metafitch_meta["country"].isna().sum()

np.int64(3999882)

In [54]:
# Remove internal node entries
metafitch_meta = metafitch_meta[~metafitch_meta["strain"].str.startswith("node_")]
metafitch_meta["country"].isna().sum()

np.int64(1542)

In [55]:
# Add country info for strains that start with hCoV-19 (added from nextclade analysis)
mask = metafitch_meta["strain"].str.startswith("hCoV-19")
metafitch_meta.loc[mask, "country"] = metafitch_meta.loc[mask, "strain"].str.split("/").str[1]
metafitch_meta["country"].isna().sum()


np.int64(974)

In [ ]:
metafitch_meta["country"] = metafitch_meta["country"].fillna("").str.strip()
metafitch_meta["strain"] = metafitch_meta["strain"].str.strip()
metafitch_meta.head(20)

In [57]:
metafitch_meta.to_csv("/Users/reem/metafitch_countries.tsv", sep="\t", index=False)

In [ ]:
# Now create date metadata
df_dates = all_nodes.merge(usher_metadata[["strain","date"]], on="strain", how="left")
df_dates.head()

In [49]:
# Extract year from date column
df_dates["date"] = df_dates["date"].str[:4]

In [74]:
df_dates.head()

,strain,date
0,Yunnan/0306-466/2020|EPI_ISL_429239|2020-03-06,2020
1,Japan/DP0803/2020|EPI_ISL_416630|2020-02-17,2020
2,England/LEED-2A8B52/2020|EPI_ISL_539074|2020-0...,2020
3,England/SHEF-C06CE/2020|EPI_ISL_420245|2020-03-25,2020
4,England/SHEF-C07F8/2020|EPI_ISL_420175|2020-03-21,2020


In [57]:
df_dates = df_dates[~df_dates["strain"].astype(str).str.startswith("node_")]
len(df_dates)

17245842

In [53]:
df_dates["date"].isna().sum()

np.int64(650)

In [71]:
mask = df_dates["date"].isna()
df_dates.loc[mask, "date"] = df_dates.loc[mask, "strain"].astype(str).str.split("/").str[2]
df_dates["date"].isna().sum()

np.int64(1)

In [ ]:
mask = df_dates["strain"].str.startswith("hCoV-19")
metafitch_meta.loc[mask, "date"] = metafitch_meta.loc[mask, "strain"].str.split("|").str[2][:4]
metafitch_meta["date"].isna().sum()

In [70]:
df_dates = pd.read_csv("/Users/reem/metafitch_dates.tsv", sep="\t")
df_dates.head()

/var/folders/bt/jpy5j9ms7pb6p6hhqzvpjffw0000gn/T/ipykernel_62322/3801401143.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dates = pd.read_csv("/Users/reem/metafitch_dates.tsv", sep="\t")


,strain,date
0,Yunnan/0306-466/2020|EPI_ISL_429239|2020-03-06,2020
1,Japan/DP0803/2020|EPI_ISL_416630|2020-02-17,2020
2,England/LEED-2A8B52/2020|EPI_ISL_539074|2020-0...,2020
3,England/SHEF-C06CE/2020|EPI_ISL_420245|2020-03-25,2020
4,England/SHEF-C07F8/2020|EPI_ISL_420175|2020-03-21,2020


In [72]:
df_dates.to_csv("/Users/reem/metafitch_dates.tsv", sep="\t", index=False)

In [58]:
# Already ran metafitch in terminal, once with the date metadata and once with the country metadata.
# Now we will merge the output files together to get a final file with both date and country metadata for each node in the tree.

countries = pd.read_csv("/Users/reem/metafitch_country_output.tsv", sep="\t")
dates = pd.read_csv("/Users/reem/metafitch_date_output.tsv", sep="\t")


/var/folders/bt/jpy5j9ms7pb6p6hhqzvpjffw0000gn/T/ipykernel_94098/2576593190.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  dates = pd.read_csv("/Users/reem/metafitch_date_output.tsv", sep="\t")


In [59]:
# Merge the two files on the strain column
merged = pd.merge(countries, dates, on="strain", how="outer")
len(merged)


21244182

In [61]:
merged["date"] = pd.to_numeric(merged["date"], errors="coerce").astype("Int64")
merged.head()

,strain,country,date
0,186951/USA/2023|PX732842.1|2023,USA,2023
1,2019_nCoV_Muc_IMB1|LR824570.1|?,Germany,<NA>
2,2020/BIKEN/A50-18|LC603287.1|?,NaN,<NA>
3,2020/BIKEN/B-1|LC603286.1|?,Japan,<NA>
4,2020/BIKEN/B-2|LC644163.1|2020-05,Japan,2020


In [65]:
merged["date"] = merged["date"].fillna(0).astype(int)
merged.head()

,strain,country,date
0,186951/USA/2023|PX732842.1|2023,USA,2023
1,2019_nCoV_Muc_IMB1|LR824570.1|?,Germany,0
2,2020/BIKEN/A50-18|LC603287.1|?,NaN,0
3,2020/BIKEN/B-1|LC603286.1|?,Japan,0
4,2020/BIKEN/B-2|LC644163.1|2020-05,Japan,2020


In [66]:
merged.to_csv("/Users/reem/merged_metafitch.tsv", sep="\t", index=False)

In [ ]:
# THE END!